In [1]:
!pip install tensorflow numba pandas numpy

In [5]:
import kagglehub
path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")

100%|██████████| 14.5M/14.5M [00:00<00:00, 93.9MB/s]

Extracting files...


In [7]:
import pandas as pd
import numpy as np
from numba import cuda
import os

@cuda.jit
def streaming_histogram(chunk, histogram, bins):
    idx = cuda.grid(1)
    if idx < chunk.size:
        value = chunk[idx]
        if value < bins:
            cuda.atomic.add(histogram, value, 1)

# Find the CSV file within the downloaded directory
files = [f for f in os.listdir(path) if f.endswith('.csv')]
if not files:
    # Sometimes datasets are in subdirectories
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            if filename.endswith('.csv'):
                csv_path = os.path.join(root, filename)
                break
else:
    csv_path = os.path.join(path, files[0])

print(f"Reading file: {csv_path}")

# Histogram Settings
NUM_BINS = 100
histogram = np.zeros(NUM_BINS, dtype=np.int32)
d_histogram = cuda.to_device(histogram)

# Load CSV in Streaming Mode using the corrected csv_path
for df in pd.read_csv(
        csv_path,
        chunksize=10000,
        encoding='ISO-8859-1'): # Online Retail dataset often needs specific encoding

    # Example Feature: Quantity
    values = df["Quantity"].fillna(0)

    # Map values into histogram bins
    values = np.clip(
        values.to_numpy(),
        0,
        NUM_BINS - 1
    ).astype(np.int32)

    d_chunk = cuda.to_device(values)

    threads = 256
    blocks = (len(values) + threads - 1) // threads

    streaming_histogram[
        blocks,
        threads
    ](
        d_chunk,
        d_histogram,
        NUM_BINS
    )

# Final Histogram
final_hist = d_histogram.copy_to_host()

print("Streaming Histogram:")
print(final_hist)

Reading file: /root/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3/online_retail_II.csv


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Streaming Histogram:
[ 22950 294346 159960  72638  72382  23165  85299   4857  25819   4094
  40956   1645 121745    935    917   1530   7907    428   3636    346
   8983    294    286    223  47036   9673    177    230    293    105
   2612     96   2449    102    102    111   7569     69     64     44
   1977     56    131     46     53     67     39     44  12162     46
   1909     30     39     29     78     26     67     29     24     19
   1459     17     19     16    582     27     46     21     14      8
     75     21   3845      8     18    100     15      4     26      8
    407     19     10     13     96     16      9      7     15     12
    112      9      4      7     11     13   3485     10     11  13832]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 29 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
